# LeetCode 1749 — Maximum Absolute Sum of Any Subarray bằng Python

Link đề: https://leetcode.com/problems/maximum-absolute-sum-of-any-subarray/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming (Kadane), Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maxAbsoluteSum(self, nums: List[int]) -> int:
        ...
```

Lộ trình: **vét cạn mọi mảng con `O(n³)`** → **cộng dồn `O(n²)`** → **hai Kadane (max và min) `O(n)`** → **tổng tiền tố: `max(P) − min(P)`** — lời giải một dòng, `O(n)` thời gian, `O(1)` bộ nhớ.

## 1. Mục tiêu học

1. Hiểu `|tổng|` lớn nhất = **max( tổng lớn nhất , −tổng nhỏ nhất )** — tách trị tuyệt đối thành hai bài con quen thuộc.
2. Viết Kadane cho cả **mảng con tổng lớn nhất** và **mảng con tổng nhỏ nhất**, cho phép mảng con **rỗng**.
3. Biến tổng mảng con thành **hiệu hai tổng tiền tố** `P[r+1] − P[l]`, và nhận ra khi lấy trị tuyệt đối thì **thứ tự** `l < r` không còn quan trọng ⇒ đáp án `max(P) − min(P)`.
4. Nhận diện các lời giải "gần đúng" phổ biến (chỉ một Kadane, quên phần tử `P[0] = 0`, lấy `abs` từng phần tử) và có phản ví dụ cụ thể.
5. Kiểm chứng bằng vét cạn ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. **Tổng tuyệt đối** của mảng con `[nums_l, …, nums_r]` là `abs(nums_l + … + nums_r)`.

Trả về **tổng tuyệt đối lớn nhất** của một mảng con bất kỳ (**có thể rỗng** — mảng con rỗng có tổng 0).

`abs(x) = -x` nếu `x < 0`, ngược lại `abs(x) = x`.

**Ví dụ 1:** `nums = [1,-3,2,3,-4]` → `5`. Mảng con `[2,3]` có `abs(2+3) = 5`.

**Ví dụ 2:** `nums = [2,-5,1,-4,3,-2]` → `8`. Mảng con `[-5,1,-4]` có `abs(-5+1-4) = 8`.

**Ràng buộc:**
- `1 ≤ nums.length ≤ 10⁵`
- `-10⁴ ≤ nums[i] ≤ 10⁴`

In [1]:
import random
import time
from itertools import accumulate
from typing import List

examples = [([1, -3, 2, 3, -4], 5), ([2, -5, 1, -4, 3, -2], 8)]
for nums, want in examples:
    print(nums, "-> kỳ vọng", want)

[1, -3, 2, 3, -4] -> kỳ vọng 5
[2, -5, 1, -4, 3, -2] -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1 Trị tuyệt đối = max của hai dấu

`abs(x) = max(x, -x)`. Do đó `max_S abs(S) = max( max_S S, max_S (−S) ) = max( max S, −min S )`. Bài toán "trị tuyệt đối lớn nhất" tách thành "lớn nhất" và "nhỏ nhất".

In [2]:
sums = [3, -8, 5, 0, -2]
print("max abs:", max(abs(s) for s in sums), "| max(max, -min):", max(max(sums), -min(sums)))

max abs: 8 | max(max, -min): 8


### 3.2 Kadane cho tổng lớn nhất (cho phép rỗng)

`cur` = tổng lớn nhất của mảng con **kết thúc tại** phần tử hiện tại, nhưng nếu âm thì "bỏ đi" (về 0 — mảng rỗng). Đáp án `best` khởi tạo 0 (mảng rỗng).

In [3]:
def kadane_max(a):
    cur = best = 0
    for x in a:
        cur = max(cur + x, 0)
        best = max(best, cur)
    return best

print(kadane_max([1, -3, 2, 3, -4]), kadane_max([-1, -2]))

5 0


### 3.3 Kadane cho tổng nhỏ nhất

Đối xứng: `cur = min(cur + x, 0)`, `best = min(best, cur)`. Hoặc chạy `kadane_max` trên mảng đổi dấu rồi đổi dấu kết quả.

In [4]:
def kadane_min(a):
    cur = best = 0
    for x in a:
        cur = min(cur + x, 0)
        best = min(best, cur)
    return best

a = [2, -5, 1, -4, 3, -2]
print(kadane_min(a), -kadane_max([-x for x in a]))

-8 -8


### 3.4 Tổng tiền tố và tổng đoạn

`P[0] = 0`, `P[k] = nums[0] + … + nums[k-1]`. Tổng `nums[l..r] = P[r+1] − P[l]`. **Mỗi cặp chỉ số `i < j` của `P`** ứng với đúng một mảng con khác rỗng; `i = j` ứng với mảng rỗng.

In [5]:
nums = [2, -5, 1, -4, 3, -2]
P = list(accumulate(nums, initial=0))
print("P =", P)
print("tổng nums[1..3] =", P[4] - P[1], "=", sum(nums[1:4]))

P = [0, 2, -3, -2, -6, -3, -5]
tổng nums[1..3] = -8 = -8


### 3.5 `max`, `min` và `accumulate` một lượt

`max(P) − min(P)` cần hai lượt duyệt `P`. Có thể duyệt một lượt, giữ `hi`, `lo` và tổng chạy `s` — không cần lưu `P`.

In [6]:
s = hi = lo = 0
for x in [2, -5, 1, -4, 3, -2]:
    s += x
    hi, lo = max(hi, s), min(lo, s)
print("hi =", hi, "lo =", lo, "-> hi - lo =", hi - lo)

hi = 2 lo = -6 -> hi - lo = 8


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `nums = [2,-5,1,-4,3,-2]`.

- Mảng con tổng **lớn nhất**: `[3]` hoặc `[2]` → 3.
- Mảng con tổng **nhỏ nhất**: `[-5,1,-4]` → −8.
- `|tổng|` lớn nhất = `max(3, 8) = 8`. ✔

Nhìn bằng tổng tiền tố: `P = [0, 2, -3, -2, -6, -3, -5]`. Tổng đoạn = hiệu hai giá trị của `P`. Hiệu **lớn nhất về trị tuyệt đối** giữa hai phần tử bất kỳ của `P` chính là `max(P) − min(P) = 2 − (−6) = 8`. Đoạn tương ứng: từ sau vị trí `P = 2` (chỉ số 1) tới vị trí `P = −6` (chỉ số 4) ⇒ `nums[1..3] = [-5,1,-4]`.

Vì có `abs`, không cần lo `max` đứng trước hay sau `min` — đổi chỗ chỉ đổi dấu tổng.

In [7]:
nums = [2, -5, 1, -4, 3, -2]
P = list(accumulate(nums, initial=0))
i_hi, i_lo = P.index(max(P)), P.index(min(P))
l, r = min(i_hi, i_lo), max(i_hi, i_lo)
print(f"max(P)={max(P)} tại {i_hi}, min(P)={min(P)} tại {i_lo} -> đoạn nums[{l}..{r-1}] = {nums[l:r]}, tổng {sum(nums[l:r])}")

max(P)=2 tại 1, min(P)=-6 tại 4 -> đoạn nums[1..3] = [-5, 1, -4], tổng -8


## 5. Cách 1 — Brute Force / baseline: mọi mảng con, tính tổng lại từ đầu (`O(n³)`)

In [8]:
def max_abs_brute(nums):
    n = len(nums)
    best = 0                              # mảng con rỗng
    for l in range(n):
        for r in range(l, n):
            best = max(best, abs(sum(nums[l:r + 1])))
    return best

for nums, want in examples:
    print(nums, "->", max_abs_brute(nums), "| kỳ vọng", want)

[1, -3, 2, 3, -4] -> 5 | kỳ vọng 5
[2, -5, 1, -4, 3, -2] -> 8 | kỳ vọng 8


**Độ phức tạp:** `O(n²)` mảng con × `O(n)` để cộng ⇒ `O(n³)`; bộ nhớ `O(n)` cho lát cắt.

**Nút thắt:** `sum(nums[l:r+1])` cộng lại các phần tử đã cộng ở vòng trước. Mở rộng `r` thêm 1 chỉ cần cộng thêm `nums[r]`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tách abs):** `max |S| = max( max S , −min S )` ⇒ hai bài Kadane.

**Insight 2 (tiền tố):** `S(l..r) = P[r+1] − P[l]` ⇒ `|S| = |P[j] − P[i]|`. Khi có trị tuyệt đối, cặp `(i, j)` **không cần** `i < j`: `|P[j] − P[i]| = |P[i] − P[j]|`. Vậy đáp án là **hiệu lớn nhất giữa hai phần tử bất kỳ của `P`** = `max(P) − min(P)`.

**Quy tắc:** *nếu* bài hỏi `|tổng đoạn|` lớn nhất *thì* đáp án = `max(P) − min(P)` với `P` **có** phần tử `0` ở đầu. *Nếu* bài hỏi tổng đoạn lớn nhất (không abs) *thì* phải giữ thứ tự: `max_{i<j} (P[j] − P[i])` — Kadane.

**Invariant khi duyệt một lượt:** sau `k` phần tử, `hi = max(P[0..k])`, `lo = min(P[0..k])`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cộng dồn khi mở rộng `r` (`O(n²)`)

In [9]:
def max_abs_quadratic(nums):
    best = 0
    for l in range(len(nums)):
        s = 0
        for r in range(l, len(nums)):
            s += nums[r]
            best = max(best, abs(s))
    return best

for nums, want in examples:
    print(max_abs_quadratic(nums), want)

5 5
8 8


**Độ phức tạp:** `O(n²)` thời gian, `O(1)` bộ nhớ. Với `n = 10⁵` là `5·10⁹` phép — quá chậm cho Python. Nút thắt: vẫn xét **mọi** cặp `(l, r)`; cần một cách tổng hợp thông tin quá khứ để không phải nhìn lại.

### 7.2 Cách 3 — Hai Kadane trong một lượt (`O(n)`)

Giữ đồng thời:
- `mx` = tổng lớn nhất của mảng con kết thúc tại đây (≥ 0, cho phép rỗng),
- `mn` = tổng nhỏ nhất của mảng con kết thúc tại đây (≤ 0).

Đáp án `max(ans, mx, -mn)` qua mọi bước.

In [10]:
def max_abs_two_kadane_demo(nums):
    mx = mn = ans = 0
    for x in nums:
        mx = max(mx + x, 0)
        mn = min(mn + x, 0)
        ans = max(ans, mx, -mn)
        print(f"x={x:>3}: mx={mx:>3}, mn={mn:>3}, ans={ans}")
    return ans

max_abs_two_kadane_demo([2, -5, 1, -4, 3, -2])

x=  2: mx=  2, mn=  0, ans=2
x= -5: mx=  0, mn= -5, ans=5
x=  1: mx=  1, mn= -4, ans=5
x= -4: mx=  0, mn= -8, ans=8
x=  3: mx=  3, mn= -5, ans=8
x= -2: mx=  1, mn= -7, ans=8


8

In [11]:
class SolutionKadane:
    def maxAbsoluteSum(self, nums: List[int]) -> int:
        mx = mn = ans = 0
        for x in nums:
            mx = max(mx + x, 0)
            mn = min(mn + x, 0)
            ans = max(ans, mx, -mn)
        return ans

for nums, want in examples:
    print(SolutionKadane().maxAbsoluteSum(nums), want)

5 5
8 8


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Tổng tiền tố: `max(P) − min(P)` (`O(n)`, gọn nhất)

In [12]:
class SolutionPrefixList:
    def maxAbsoluteSum(self, nums: List[int]) -> int:
        P = list(accumulate(nums, initial=0))
        return max(P) - min(P)

class SolutionPrefix:
    def maxAbsoluteSum(self, nums: List[int]) -> int:
        s = hi = lo = 0
        for x in nums:
            s += x
            if s > hi: hi = s
            if s < lo: lo = s
        return hi - lo

for nums, want in examples:
    print(SolutionPrefixList().maxAbsoluteSum(nums), SolutionPrefix().maxAbsoluteSum(nums), want)

5 5 5
8 8 8


**Độ phức tạp:** bản list `O(n)` thời gian, `O(n)` bộ nhớ (lưu `P`); bản một lượt `O(n)` thời gian, `O(1)` bộ nhớ. Vì `hi ≥ P[0] = 0 ≥ lo` nên kết quả luôn ≥ 0 — khớp với "mảng con có thể rỗng".

### 7.4 Cách 5 — Chia để trị bằng cách ghép bộ giá trị (góc nhìn bổ sung)

Cách này không tối ưu bằng Cách 4 nhưng là bài luyện **chia để trị** kinh điển (giống "Maximum Subarray" bản D&C). Với đoạn `[lo, hi)`, ta trả về bộ 5 giá trị:

- `total` — tổng cả đoạn,
- `pre_max`, `pre_min` — tổng tiền tố lớn nhất / nhỏ nhất của đoạn (cho phép rỗng ⇒ so với 0),
- `suf_max`, `suf_min` — tổng hậu tố lớn nhất / nhỏ nhất,
- `best_max`, `best_min` — tổng mảng con lớn nhất / nhỏ nhất **nằm trong** đoạn.

Ghép hai nửa `A | B`:
- mảng con tốt nhất hoặc nằm gọn trong `A`, hoặc gọn trong `B`, hoặc **cắt qua giữa** = `suf(A) + pre(B)`;
- `pre(A|B) = max(pre(A), total(A) + pre(B))`, tương tự cho hậu tố.

Đây chính là "nút" của **segment tree** cho bài toán truy vấn đoạn — nếu bài có thêm cập nhật phần tử thì cách này trở thành cách duy nhất khả thi.

In [13]:
def merge(A, B):
    tA, pmaxA, pminA, smaxA, sminA, bmaxA, bminA = A
    tB, pmaxB, pminB, smaxB, sminB, bmaxB, bminB = B
    return (
        tA + tB,
        max(pmaxA, tA + pmaxB), min(pminA, tA + pminB),
        max(smaxB, tB + smaxA), min(sminB, tB + sminA),
        max(bmaxA, bmaxB, smaxA + pmaxB),
        min(bminA, bminB, sminA + pminB),
    )

def leaf(x):
    return (x, max(x, 0), min(x, 0), max(x, 0), min(x, 0), max(x, 0), min(x, 0))

def solve_dc(nums, lo, hi):
    if hi - lo == 1:
        return leaf(nums[lo])
    mid = (lo + hi) // 2
    return merge(solve_dc(nums, lo, mid), solve_dc(nums, mid, hi))

class SolutionDC:
    def maxAbsoluteSum(self, nums: List[int]) -> int:
        node = solve_dc(nums, 0, len(nums))
        return max(node[5], -node[6])

for nums, want in examples:
    print(SolutionDC().maxAbsoluteSum(nums), want, "| nút gốc:", solve_dc(nums, 0, len(nums)))

5 5 | nút gốc: (-1, 3, -2, 1, -4, 5, -4)
8 8 | nút gốc: (-5, 2, -6, 1, -7, 3, -8)


**Độ phức tạp:** mỗi lần ghép `O(1)` nên `T(n) = 2T(n/2) + O(1) = O(n)`; tuy vậy chi phí gọi hàm đệ quy trong Python khá lớn (chậm hơn Cách 4 nhiều lần). Độ sâu đệ quy chỉ `log₂ n ≈ 17` nên không lo giới hạn đệ quy. (Bản D&C "cổ điển" tính đoạn cắt qua giữa bằng vòng lặp sẽ là `O(n log n)`; bản ghép bộ giá trị như trên tránh được điều đó.) Bộ nhớ `O(log n)` cho stack.

## 8. TRACE từng bước

Trace Cách 4 (một lượt) trên Ví dụ 1: `nums = [1,-3,2,3,-4]`.

| bước | x | `s` (= P) | `hi` | `lo` | `hi − lo` |
|---|---|---|---|---|---|
| 0 | — | 0 | 0 | 0 | 0 |
| 1 | 1 | 1 | 1 | 0 | 1 |
| 2 | -3 | -2 | 1 | -2 | 3 |
| 3 | 2 | 0 | 1 | -2 | 3 |
| 4 | 3 | 3 | 3 | -2 | **5** |
| 5 | -4 | -1 | 3 | -2 | 5 |

`hi = 3` ở bước 4 (`P[4]`), `lo = −2` ở bước 2 (`P[2]`) ⇒ đoạn `nums[2..3] = [2, 3]`, tổng 5. ✔

In [14]:
def trace_prefix(nums):
    s = hi = lo = 0
    i_hi = i_lo = 0
    print(f"bước 0: s=0 hi=0 lo=0")
    for k, x in enumerate(nums, start=1):
        s += x
        if s > hi: hi, i_hi = s, k
        if s < lo: lo, i_lo = s, k
        print(f"bước {k}: x={x:>3} s={s:>3} hi={hi:>3} (P[{i_hi}]) lo={lo:>3} (P[{i_lo}]) hi-lo={hi-lo}")
    l, r = min(i_hi, i_lo), max(i_hi, i_lo)
    print(f"=> đáp án {hi - lo}, đoạn nums[{l}..{r-1}] = {nums[l:r]}")
    return hi - lo

trace_prefix([1, -3, 2, 3, -4])

bước 0: s=0 hi=0 lo=0
bước 1: x=  1 s=  1 hi=  1 (P[1]) lo=  0 (P[0]) hi-lo=1
bước 2: x= -3 s= -2 hi=  1 (P[1]) lo= -2 (P[2]) hi-lo=3
bước 3: x=  2 s=  0 hi=  1 (P[1]) lo= -2 (P[2]) hi-lo=3
bước 4: x=  3 s=  3 hi=  3 (P[4]) lo= -2 (P[2]) hi-lo=5
bước 5: x= -4 s= -1 hi=  3 (P[4]) lo= -2 (P[2]) hi-lo=5
=> đáp án 5, đoạn nums[2..3] = [2, 3]


5

Trace Cách 3 trên Ví dụ 2 dưới dạng bảng (để thấy `mn` tích lũy đoạn âm `[-5, 1, -4]` mà không bị reset):

| x | `mx = max(mx + x, 0)` | `mn = min(mn + x, 0)` | `ans` |
|---|---|---|---|
| 2 | 2 | 0 | 2 |
| -5 | 0 | -5 | 5 |
| 1 | 1 | -4 | 5 |
| -4 | 0 | -8 | **8** |
| 3 | 3 | -5 | 8 |
| -2 | 1 | -7 | 8 |

Chú ý `mn` sau `1` là `-4` (không reset vì vẫn âm) — đúng tinh thần "giữ tiền tố có lợi cho mục tiêu nhỏ nhất".

Trace Cách 3 trên Ví dụ 1 để so sánh: `mx` "bắt" được `[2,3]`, còn `mn` bắt `[-3]` hoặc `[-4]`.

In [15]:
max_abs_two_kadane_demo([1, -3, 2, 3, -4])

x=  1: mx=  1, mn=  0, ans=1
x= -3: mx=  0, mn= -3, ans=3
x=  2: mx=  2, mn= -1, ans=3
x=  3: mx=  5, mn=  0, ans=5
x= -4: mx=  1, mn= -4, ans=5


5

## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def maxAbsoluteSum(self, nums: List[int]) -> int:
        s = hi = lo = 0               # hi = max tiền tố, lo = min tiền tố (có P[0] = 0)
        for x in nums:
            s += x
            if s > hi:
                hi = s
            elif s < lo:
                lo = s
        return hi - lo

for nums, want in examples:
    assert Solution().maxAbsoluteSum(nums) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


(`elif` an toàn vì một giá trị `s` không thể vừa lớn hơn `hi` vừa nhỏ hơn `lo` — `hi ≥ lo` luôn đúng.)

## 10. Vì sao thuật toán đúng

**Mệnh đề:** `max_{mảng con A} |sum(A)| = max(P) − min(P)` với `P[0..n]` là tổng tiền tố có `P[0] = 0`.

*Chiều ≤:* mọi mảng con `nums[l..r]` có tổng `P[r+1] − P[l]` ⇒ `|P[r+1] − P[l]| ≤ max(P) − min(P)` (hiệu hai phần tử bất kỳ không vượt quá hiệu max–min). Mảng rỗng có tổng 0 ≤ `max(P) − min(P)`.

*Chiều ≥:* gọi `a` là chỉ số đạt `max(P)`, `b` là chỉ số đạt `min(P)`. Nếu `a = b` thì `max = min` (mọi `P` bằng nhau), đáp án 0 đạt bằng mảng rỗng. Nếu `a ≠ b`, đặt `i = min(a, b)`, `j = max(a, b)`: mảng con `nums[i..j-1]` có tổng `P[j] − P[i] = ±(max(P) − min(P))`, trị tuyệt đối đúng bằng `max(P) − min(P)`.

**Cách 3 đúng** vì `mx` sau mỗi bước = max tổng mảng con (có thể rỗng) kết thúc tại đây (Kadane chuẩn), tương tự `mn`; lấy max qua mọi bước ra `max S` và `−min S`.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn + `sum` | `O(n³)` | `O(n)` |
| 2. Cộng dồn | `O(n²)` | `O(1)` |
| 3. Hai Kadane | `O(n)` | `O(1)` |
| 4a. `max(P) − min(P)` (list) | `O(n)` | `O(n)` |
| 4b. Tiền tố một lượt | `O(n)` | `O(1)` |
| 5. Chia để trị | `O(n)` (hằng số lớn) | `O(log n)` stack |

Giá trị lớn nhất có thể: `10⁵ · 10⁴ = 10⁹` — vừa `int32` (`< 2.147·10⁹`). Đo thực nghiệm:

In [17]:
rng = random.Random(1749)
big = [rng.randint(-10**4, 10**4) for _ in range(10**5)]
for name, sol in [("Cách 3", SolutionKadane()), ("Cách 4a", SolutionPrefixList()), ("Cách 4b/Nộp", Solution()), ("Cách 5 D&C", SolutionDC())]:
    t = time.perf_counter()
    r = sol.maxAbsoluteSum(big)
    print(f"{name}: {r} trong {1000*(time.perf_counter()-t):.1f} ms")
small = big[:2000]
t = time.perf_counter(); max_abs_quadratic(small); dt = time.perf_counter() - t
print(f"Cách 2 với n=2000: {dt:.2f}s -> n=10^5 ước tính ≈ {dt * 2500:.0f}s")
print("worst:", Solution().maxAbsoluteSum([-10**4] * 10**5))

Cách 3: 5130587 trong 9.1 ms
Cách 4a: 5130587 trong 5.6 ms
Cách 4b/Nộp: 5130587 trong 3.6 ms
Cách 5 D&C: 5130587 trong 54.3 ms


Cách 2 với n=2000: 0.11s -> n=10^5 ước tính ≈ 267s
worst: 1000000000


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
sols = [Solution(), SolutionKadane(), SolutionPrefixList(), SolutionPrefix(), SolutionDC()]
edge = examples + [
    ([0], 0),
    ([5], 5),
    ([-5], 5),                      # một phần tử âm
    ([1, -1], 1),
    ([-1, -1], 2),                  # toàn âm: lấy cả mảng
    ([3, 3, 3], 9),
    ([10**4, -10**4] * 3, 10**4),
    ([-10**4] * 10, 10**5),
]
for nums, want in edge:
    for s in sols:
        assert s.maxAbsoluteSum(nums) == want, (nums, s)
    assert max_abs_brute(nums) == want == max_abs_quadratic(nums)
    print("OK", nums[:8], "->", want)

OK [1, -3, 2, 3, -4] -> 5
OK [2, -5, 1, -4, 3, -2] -> 8
OK [0] -> 0
OK [5] -> 5
OK [-5] -> 5
OK [1, -1] -> 1
OK [-1, -1] -> 2
OK [3, 3, 3] -> 9
OK [10000, -10000, 10000, -10000, 10000, -10000] -> 10000
OK [-10000, -10000, -10000, -10000, -10000, -10000, -10000, -10000] -> 100000


### 12.2 Ngẫu nhiên có seed, đối chiếu vét cạn

In [19]:
rng = random.Random(77)
for _ in range(1000):
    n = rng.randint(1, 12)
    lim = rng.choice([1, 3, 10, 10**4])
    nums = [rng.randint(-lim, lim) for _ in range(n)]
    want = max_abs_brute(nums)
    for s in sols:
        assert s.maxAbsoluteSum(nums) == want, nums
for _ in range(10):
    nums = [rng.randint(-10**4, 10**4) for _ in range(rng.randint(100, 600))]
    assert len({s.maxAbsoluteSum(nums) for s in sols} | {max_abs_quadratic(nums)}) == 1
print("1000 case nhỏ + 10 case vừa: khớp")

1000 case nhỏ + 10 case vừa: khớp


### 12.3 Kiểm tra tính chất: đáp án = max(Kadane max, −Kadane min)

In [20]:
rng = random.Random(8)
for _ in range(500):
    nums = [rng.randint(-20, 20) for _ in range(rng.randint(1, 30))]
    assert Solution().maxAbsoluteSum(nums) == max(kadane_max(nums), -kadane_min(nums))
print("tính chất tách abs đúng trên 500 case")

tính chất tách abs đúng trên 500 case


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ chạy Kadane tổng lớn nhất rồi lấy `abs`

In [21]:
def wrong_only_max(nums):
    return abs(kadane_max(nums))           # SAI: bỏ qua đoạn có tổng rất âm

nums = [2, -5, 1, -4, 3, -2]
print("sai:", wrong_only_max(nums), "| đúng:", Solution().maxAbsoluteSum(nums))

sai: 3 | đúng: 8


**Vì sao sai:** `|tổng|` lớn có thể đến từ đoạn **âm** sâu (`[-5,1,-4] = −8`). Kadane max chỉ thấy 3. **Sửa:** chạy thêm Kadane min, hoặc dùng `max(P) − min(P)`.

### Lỗi 2 — Lấy trị tuyệt đối từng phần tử rồi mới tìm tổng

In [22]:
def wrong_abs_each(nums):
    return kadane_max([abs(x) for x in nums])   # SAI: = tổng mọi |x|

nums = [1, -1]
print("sai:", wrong_abs_each(nums), "| đúng:", Solution().maxAbsoluteSum(nums))

sai: 2 | đúng: 1


**Vì sao sai:** `abs(1 + (-1)) = 0`, không phải `|1| + |-1| = 2`. Trị tuyệt đối áp lên **tổng**, không áp lên từng phần tử (`|a + b| ≤ |a| + |b|`). Đáp án đúng `1` (đoạn `[1]` hoặc `[-1]`).

### Lỗi 3 — Quên `P[0] = 0` trong cách tiền tố

In [23]:
def wrong_no_zero(nums):
    P = list(accumulate(nums))             # SAI: thiếu 0 đầu
    return max(P) - min(P)

for nums in ([5], [-3], [1, 2, 3]):
    print(nums, "sai:", wrong_no_zero(nums), "| đúng:", Solution().maxAbsoluteSum(nums))

[5] sai: 0 | đúng: 5
[-3] sai: 0 | đúng: 3
[1, 2, 3] sai: 5 | đúng: 6


**Vì sao sai:** không có `P[0] = 0` thì mất các mảng con **bắt đầu ở chỉ số 0** (tổng `P[j] − P[0]`). Với `[5]`, `P = [5]` ⇒ `5 − 5 = 0`, trong khi đáp án là 5. **Sửa:** `accumulate(nums, initial=0)` hoặc khởi tạo `hi = lo = 0`.

### Lỗi 4 — Một Kadane duy nhất, lấy `abs(cur)`

In [24]:
def wrong_abs_cur(nums):
    cur, best = 0, 0
    for x in nums:
        cur = max(cur, 0) + x               # Kadane MAX ...
        best = max(best, abs(cur))          # SAI: ... nhưng cur bị reset khi âm
    return best

nums = [-1, -1]
print("sai:", wrong_abs_cur(nums), "| đúng:", Solution().maxAbsoluteSum(nums))

sai: 1 | đúng: 2


**Vì sao sai:** Kadane max **cắt bỏ** phần tiền tố âm (`max(cur, 0)`), nên không bao giờ tích lũy được đoạn âm dài `[-1,-1]`. Hai mục tiêu ngược chiều cần **hai** biến trạng thái riêng. **Sửa:** Cách 3.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Thấy `|tổng đoạn|`** ⇒ nghĩ ngay `max(P) − min(P)` (với `P[0] = 0`). Đây là một trong những "mẹo một dòng" đẹp nhất về tổng tiền tố.
- **Tổng đoạn lớn nhất (không abs)** ⇒ Kadane, hoặc `max_j (P[j] − min_{i<j} P[i])` — phải giữ thứ tự.
- `abs` của max/min ⇒ tách thành **hai bài** (max và min) rồi lấy lớn hơn.
- "Mảng con có thể rỗng" ⇒ khởi tạo đáp án 0 / `hi = lo = 0`.
- Kiểm tra nhanh bằng ví dụ 1 phần tử âm (`[-5]` → 5) và toàn âm.

## 15. Biến thể của bài

1. **LeetCode 53 — Maximum Subarray:** chỉ tổng lớn nhất (mảng con khác rỗng) — Kadane.
2. **Độ dài ít nhất `k`:** `max_j max(P[j] − min_{i ≤ j−k} P[i], max_{i ≤ j−k} P[i] − P[j])` — giữ min/max tiền tố "trễ" `k` bước (Bài tập 5).
3. **Mảng vòng:** tổng tuyệt đối lớn nhất của mảng con trên mảng vòng (kết hợp LeetCode 918).
4. **Tổng đoạn gần `0` nhất** (min `|tổng|` với đoạn khác rỗng): sắp xếp `P` rồi xét hiệu các phần tử kề nhau — `O(n log n)`.
5. **Trả về chính đoạn** đạt đáp án: lưu chỉ số của `hi`, `lo` (đã làm ở mục 8).

Minh họa biến thể "có cập nhật": dùng lại `merge`/`leaf` của Cách 5 trong một **segment tree** đơn giản — mỗi lần gán `nums[i] = v` rồi hỏi lại đáp án chỉ tốn `O(log n)`. (Ví dụ tự dựng, đối chiếu bằng `Solution` chạy lại từ đầu.)

In [25]:
class AbsSumSegTree:
    def __init__(self, nums):
        self.n = len(nums)
        self.size = 1
        while self.size < self.n:
            self.size *= 2
        empty = (0, 0, 0, 0, 0, 0, 0)          # đoạn rỗng: mọi giá trị = 0
        self.t = [empty] * (2 * self.size)
        for i, x in enumerate(nums):
            self.t[self.size + i] = leaf(x)
        for i in range(self.size - 1, 0, -1):
            self.t[i] = merge(self.t[2 * i], self.t[2 * i + 1])

    def update(self, i, v):
        i += self.size
        self.t[i] = leaf(v)
        i //= 2
        while i:
            self.t[i] = merge(self.t[2 * i], self.t[2 * i + 1])
            i //= 2

    def answer(self):
        return max(self.t[1][5], -self.t[1][6])

rng = random.Random(4)
arr = [rng.randint(-10, 10) for _ in range(13)]
st = AbsSumSegTree(arr)
for _ in range(300):
    i, v = rng.randrange(len(arr)), rng.randint(-10, 10)
    arr[i] = v; st.update(i, v)
    assert st.answer() == Solution().maxAbsoluteSum(arr)
print("segment tree + 300 cập nhật: khớp; đáp án cuối =", st.answer())

segment tree + 300 cập nhật: khớp; đáp án cuối = 25


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi `(l, r)`, `sum` lại | `O(n³)` | `O(n)` | chuẩn đối chiếu |
| 2. Cộng dồn | mở rộng `r`, `abs(s)` | `O(n²)` | `O(1)` | `n` ≤ vài nghìn |
| 3. Hai Kadane | `max(mx, −mn)` | `O(n)` | `O(1)` | khi đã quen Kadane; mở rộng được sang biến thể có thao tác |
| 4. Tiền tố | `max(P) − min(P)` | `O(n)` | `O(1)` | **nộp bài**, ngắn nhất |
| 5. Chia để trị | ghép bộ (tổng, tiền tố, hậu tố, tốt nhất) | `O(n)` phép ghép | `O(log n)` | nền cho segment tree khi có cập nhật |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `maxAbsoluteSum([-1, 2, -1])` và `maxAbsoluteSum([4, -4, 4, -4])` bằng bao nhiêu?

**Bài 2 (tính tay):** Viết mảng `P` cho Ví dụ 2 và chỉ ra `max(P)`, `min(P)`, đoạn tương ứng.

**Bài 3 (điền TODO):** Viết `best_segment(nums)` trả về `(giá_trị, l, r)` với `nums[l:r]` là một đoạn đạt đáp án (`l == r` nếu đáp án 0 – mảng rỗng).

**Bài 4 (tự viết lại):** Viết lại Cách 3 nhưng **chỉ dùng một hàm Kadane max**, áp lên `nums` và `[-x for x in nums]`.

**Bài 5 (mini challenge):** Viết `max_abs_len_at_least(nums, k)` — tổng tuyệt đối lớn nhất của mảng con có **độ dài ≥ k** (`1 ≤ k ≤ n`), chạy `O(n)`.

In [26]:
def best_segment(nums):
    # TODO
    pass

def max_abs_via_one_kadane(nums):
    # TODO
    pass

def max_abs_len_at_least(nums, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Tính `P` | `[-1,2,-1]`: `P = [0,-1,1,0]` | `[4,-4,4,-4]`: `P` dao động 0 ↔ 4 |
| 2 | `P[0] = 0` | `P = [0,2,-3,-2,-6,-3,-5]` | max 2 tại 1, min −6 tại 4 |
| 3 | Lưu chỉ số đạt `hi`, `lo` | `l = min(i_hi, i_lo)`, `r = max(...)` | Khởi tạo cả hai chỉ số = 0 |
| 4 | `−min S = max(−S)` | `max(kadane_max(a), kadane_max(neg))` | Hàm Kadane cho phép rỗng |
| 5 | Đoạn `nums[i..j-1]` có độ dài `j − i ≥ k` | Duyệt `j` từ `k` tới `n`, đưa `P[j−k]` vào min/max chạy | `ans = max(ans, P[j] − mn, mx − P[j])` |

## 19. Đáp án tham khảo

In [27]:
print("Bài 1:", Solution().maxAbsoluteSum([-1, 2, -1]), Solution().maxAbsoluteSum([4, -4, 4, -4]))
assert Solution().maxAbsoluteSum([-1, 2, -1]) == 2
assert Solution().maxAbsoluteSum([4, -4, 4, -4]) == 4
P2 = list(accumulate([2, -5, 1, -4, 3, -2], initial=0))
print("Bài 2: P =", P2, "max", max(P2), "min", min(P2))
assert max(P2) - min(P2) == 8

Bài 1: 2 4
Bài 2: P = [0, 2, -3, -2, -6, -3, -5] max 2 min -6


In [28]:
def best_segment(nums):
    s = hi = lo = 0
    i_hi = i_lo = 0
    for k, x in enumerate(nums, start=1):
        s += x
        if s > hi: hi, i_hi = s, k
        if s < lo: lo, i_lo = s, k
    l, r = min(i_hi, i_lo), max(i_hi, i_lo)
    return hi - lo, l, r

def max_abs_via_one_kadane(nums):
    return max(kadane_max(nums), kadane_max([-x for x in nums]))

rng = random.Random(31)
for _ in range(500):
    nums = [rng.randint(-9, 9) for _ in range(rng.randint(1, 12))]
    v, l, r = best_segment(nums)
    assert v == max_abs_brute(nums) and abs(sum(nums[l:r])) == v
    assert max_abs_via_one_kadane(nums) == v
print("Bài 3, 4: OK", best_segment([2, -5, 1, -4, 3, -2]))

Bài 3, 4: OK (8, 1, 4)


In [29]:
def max_abs_len_at_least(nums, k):
    n = len(nums)
    P = list(accumulate(nums, initial=0))
    mn = float("inf"); mx = float("-inf"); ans = 0
    for j in range(k, n + 1):
        mn = min(mn, P[j - k]); mx = max(mx, P[j - k])
        ans = max(ans, P[j] - mn, mx - P[j])
    return ans

def len_at_least_brute(nums, k):
    n = len(nums)
    return max(abs(sum(nums[l:r])) for l in range(n) for r in range(l + k, n + 1))

rng = random.Random(55)
for _ in range(500):
    nums = [rng.randint(-9, 9) for _ in range(rng.randint(1, 12))]
    k = rng.randint(1, len(nums))
    assert max_abs_len_at_least(nums, k) == len_at_least_brute(nums, k), (nums, k)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK, ví dụ:", max_abs_len_at_least([1, -3, 2, 3, -4], 3))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK, ví dụ: 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
s = hi = lo = 0                 # P[0] = 0 nằm sẵn trong hi, lo
for x in nums:
    s += x
    hi = max(hi, s)
    lo = min(lo, s)
return hi - lo                  # |tổng đoạn| lớn nhất
```

**Checklist:**
- [ ] Có tính `P[0] = 0` (khởi tạo `hi = lo = 0`)?
- [ ] Xét cả đoạn tổng **âm** (không chỉ Kadane max)?
- [ ] `abs` áp lên **tổng**, không lên từng phần tử?
- [ ] Kiểm `[−5]` → 5, `[1, −1]` → 1, `[−1, −1]` → 2?
- [ ] `O(n)` thời gian, `O(1)` bộ nhớ.